# 01 — Exploratory Data Analysis (EDA)

**Corpus**: 227 listing kos REAL hasil scrape Mamikos.com (deskripsi pemilik + koordinat asli), 18 kecamatan Bandar Lampung.

**Tujuan**: pahami distribusi data sebelum preprocessing + indexing untuk informasi Dataset 15% rubric.

## Setup

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
# Sumber real committed (raw listing fields). Untuk corpus terindeks: data/processed/corpus.json
DATA_PATH = Path('../data/raw/mamikos_real_v2.jsonl')

## Load Corpus

In [ ]:
with open(DATA_PATH, encoding='utf-8') as f:
    data = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(data)
print(f'Total listings: {len(df)}')
df.head(3)

## 1. Word Count Distribution (deskripsi pemilik asli — terse)

In [ ]:
df['deskripsi'] = df['deskripsi'].fillna('')
df['word_count'] = df['deskripsi'].str.split().str.len()
print(df['word_count'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['word_count'], bins=25, color='#2563eb', edgecolor='black')
ax.axvline(df['word_count'].median(), color='red', linestyle='--', label=f'Median: {int(df["word_count"].median())} kata')
ax.set_xlabel('Word count per deskripsi (cerita pemilik)')
ax.set_ylabel('Listings')
ax.set_title(f'Distribusi panjang deskripsi real (n={len(df)})')
ax.legend()
plt.tight_layout()
plt.savefig('charts/01_word_count.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. Price Distribution

In [ ]:
print(df['harga_per_bulan'].describe())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['harga_per_bulan'], bins=30, color='#047857', edgecolor='black')
ax.axvline(df['harga_per_bulan'].median(), color='red', linestyle='--', label=f'Median: Rp {int(df["harga_per_bulan"].median()):,}')
ax.set_xlabel('Harga per bulan (IDR)')
ax.set_ylabel('Listings')
ax.set_title('Distribusi harga sewa kos real (Rp 300k - 6jt)')
ax.legend()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))
plt.tight_layout()
plt.savefig('charts/01_price.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Tipe Distribution

In [ ]:
tipe_counts = df['tipe'].value_counts()
print(tipe_counts)

fig, ax = plt.subplots(figsize=(7, 4))
colors = {'putra': '#2563eb', 'putri': '#ec4899', 'campur': '#8b5cf6'}
tipe_counts.plot(kind='bar', ax=ax, color=[colors.get(t, '#999') for t in tipe_counts.index])
ax.set_title('Distribusi tipe kos real (campur/putri/putra)')
ax.set_ylabel('Count')
ax.set_xlabel('Tipe')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('charts/01_tipe.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Kecamatan Coverage (18 kecamatan dengan data real)

In [ ]:
kec_counts = df['kecamatan'].value_counts()
print(f'Total kecamatan unique: {len(kec_counts)}')

fig, ax = plt.subplots(figsize=(8, 8))
kec_counts.sort_values().plot(kind='barh', ax=ax, color='#0891b2')
ax.set_title('Distribusi listings per kecamatan Bandar Lampung')
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig('charts/01_kecamatan.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Fasilitas Frequency

In [ ]:
fasilitas_flat = [f for fs in df['fasilitas'] for f in fs]
fac_counts = Counter(fasilitas_flat)
print('Top 10 fasilitas:')
for fac, cnt in fac_counts.most_common(10):
    print(f'  {fac}: {cnt}')

top_fac = pd.DataFrame(fac_counts.most_common(15), columns=['fasilitas', 'count'])
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_fac['fasilitas'][::-1], top_fac['count'][::-1], color='#7c3aed')
ax.set_title('Top 15 fasilitas')
ax.set_xlabel('Listings yang punya fasilitas ini')
plt.tight_layout()
plt.savefig('charts/01_fasilitas.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Universitas Mentions (Multi-University Coverage)

In [ ]:
UNIV_ALIASES = {
    'UNILA': ['UNILA', 'Universitas Lampung', 'unyila', 'Unila'],
    'Polinela': ['Polinela', 'Politeknik Negeri Lampung'],
    'IBI Darmajaya': ['IBI Darmajaya', 'Darmajaya'],
    'UBL': ['UBL', 'Universitas Bandar Lampung'],
    'UIN Raden Intan': ['UIN Raden Intan', 'UIN RIL', 'UIN Lampung'],
    'Teknokrat': ['Teknokrat', 'Universitas Teknokrat'],
    'Malahayati': ['Malahayati', 'Universitas Malahayati'],
    'ITERA': ['ITERA', 'Institut Teknologi Sumatera'],
    'Saburai': ['Saburai', 'Universitas Saburai'],
}

univ_counts = {}
for univ, aliases in UNIV_ALIASES.items():
    count = df['deskripsi'].apply(lambda d: any(a in d for a in aliases)).sum()
    univ_counts[univ] = count

univ_df = pd.Series(univ_counts).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
univ_df.plot(kind='barh', ax=ax, color='#dc2626')
ax.set_title('Mentions universitas di deskripsi listing')
ax.set_xlabel('Count listings')
plt.tight_layout()
plt.savefig('charts/01_universities.png', dpi=100, bbox_inches='tight')
plt.show()

## Insights (data real)

1. **18 kecamatan** ter-cover; dominasi Sukarame/Kedaton/Rajabasa karena density kos Mamikos tertinggi di sekitar UNILA/UIN/ITERA/Polinela — mencerminkan pasar nyata.
2. **Harga real**: median ~Rp 800k, range Rp 300k - 6jt.
3. **Tipe**: campur 50%, putri 37%, putra 13% (tidak ada pasutri — Mamikos gender 0/1/2 saja).
4. **Deskripsi terse**: median ~23 kata, cerita pemilik asli (bukan padded). Lebih pendek dari synthetic tapi otentik.
5. **Universitas mentions**: UNILA/Teknokrat/Polinela/UIN paling sering disebut pemilik karena memang area kos terpadat.

Detail di [LAPORAN.md](../LAPORAN.md) section 3.